# Spatial multi-omics integration


Spatial multi-omics, can give you insights on how different molecules might interact together in space. We will therefore have a look at the Vicari et al., 2024 dataset, which applies mass spectrometry and Visium on the same slice. The experiments were conducted on mouse brain which were unilaterally treated with 6'OHDA, causing dopaminergic neuron depletion on the treated site. For more information check out the [paper by Vicari et al., 2024](https://www.nature.com/articles/s41587-023-01937-y).

In this session, you will learn:
   1) how spatial multi-omics measurement can look like from the data perspective
   2) Look at potential problems of spatial multi-omics measurements 
   3) Learn various downstream analysis approaches on this type of data (separate vs. joint). Going from NMF separate analysis to principled joint analysis using MOFA/MEFISTO

**References**:
* [Data: Vicari et al., 2024](https://www.nature.com/articles/s41587-023-01937-y).
* [MAGPIE Williams et al., 2026](https://www.nature.com/articles/s41467-025-68003-w)
* https://biofam.github.io/MOFA2/
* https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.NMF.html#sklearn.decomposition.NMF

In [ ]:
import numpy as np
import scanpy as sc
import pandas as pd
import mofax as mfx
import matplotlib.pyplot as plt


from sklearn.decomposition import NMF
from scipy.stats import pearsonr
from mofapy2.run.entry_point import entry_point

import warnings

warnings.filterwarnings("ignore")

In [ ]:
# set seed
seed = 42

## Load the Vicari et al., 2024 spatial multi-omics data.

We will use sample: ```V11L12-109_B1``` in this tutorial. It is a mouse brain from the substantia nigra region, which was unilaterly treated with 6'OHDA causing dopaminergic neuron depletion at the site of action. Mass spectrometry (MALDI) and Visium were conducted consecutively on the same slide, respectively, creating two coordinate systems that need to be aligned. The data provided in this tutorial was already aligned using [MAGPIE](https://www.nature.com/articles/s41467-025-68003-w), which is a landmark based alignment tool. The generated data contained only one H&E image, in MAGPIE you can align based on the principal components of the modality with missing H&E. Additionally, MAGPIE provides aggregation/ 1:1 matching of observations, by matching mass spectrometry observations to Visium spots. For more information, check out the [MAGPIE paper](https://www.nature.com/articles/s41467-025-68003-w).

Inside the ```V11L12-109_B1``` folder, you will find three datasets: the Visium data (```visium```), the raw mass spectrometry data without aggregation (```msi```), and the 1:1 matched mass spectrometry data (```msi_aggregated```). The MAGPIE workflow formats the mass spectrometry data so that it can be accessed using the standard Visium read functions.

In [ ]:
visium = sc.read_visium("data/V11L12-109_B1/visium/", library_id="rna")
msi = sc.read_visium("data/V11L12-109_B1/msi_aggregated/", library_id="msi")
print(visium, msi)

You may have noticed that the Visium dataset contains many more observations than the MSI dataset. However, after performing a 1:1 matching, both should contain the same set of observations. To achieve this, we need to manually take the intersection of the two datasets. We can also library normalize, log-scale the data and derive highly variable features.

In [ ]:
msi_barcodes = set(msi.obs_names)
visium_barcodes = set(visium.obs_names)

intersection = set.intersection(msi_barcodes, visium_barcodes)

visium = visium[list(intersection)]
msi = msi[list(intersection)]

# save raw data
visium.layers["counts"] = visium.X.copy()
sc.pp.normalize_total(visium, target_sum=1e4)
sc.pp.log1p(visium)

msi.layers["raw"] = msi.X.copy()
sc.pp.normalize_total(msi, target_sum=1e4)
sc.pp.log1p(msi)

In [ ]:
# feature selection
# highly variable features
sc.pp.highly_variable_genes(visium, n_top_genes=2000, subset=True)
sc.pp.highly_variable_genes(msi, n_top_genes=500, subset=True)

print(visium, msi)

In scanpy, the function sc.pl.spatial is very convenient. You simply provide a feature name—stored in .obs—and it will plot that feature overlaid on the H&E image. From the supplementary information, we know that ```mz-390.16864``` can be assigned to Dopamine. We can plot the Dopamine signal and look at the distribution in space.

In [ ]:
sc.pl.spatial(msi, img_key="hires", color="mz-390.16864", alpha=0.8)

As expected, we can see an unilateral distribution of Dopamine. 

**Task 1**: Plot the gene ```Penk``` for the Visium data using the ```sc.pl.spatial``` function. What is the function of Penk and how is connected to Dopamine?

## Learning spatial patterns of variation

One of the key tasks in spatial omics analysis is identifying observations with similar expression profiles. In spatial data, this translates into detecting spatial domains or patterns, where groups of observations share similar expresion. In this section, we will use two main approaches to integrate multi-modal data. We will learn how to look at the omics-layers *separately* but still try to connect these and look at them at joint.

We will work with non-negative matrix factorization (NMF) and [MOFA](https://www.embopress.org/doi/full/10.15252/msb.20178124), two of most common used tools for any downstream analysis. 

We will first apply NMF separately on both modalities. Since we already know the distribution of Dopamine (mz-390.16864), we can initially apply NMF to the Visium dataset and compute the cross-correlation between each NMF factor and the Dopamine distribution.

We will:

1. Visualize the learned latent embeddings (spatial patterns).
2. Compute cross-correlations, focusing on NMF factors that show strong correlation with Dopamine.
3. Visualize the dopamine distribution alongside the most correlated NMF factors and their top loadings.
4. Compare the patterns across both modalities to check whether similar spatial patterns are learned.
5. Optional: then we will go one step into joint learning by trying the naive approach by concatenating both modalities by their feature axis and run NMF.

In [ ]:
# NMF factorization
n_factors = 15  # you can play around with number of latent factors throughout the tutorial
nmf = NMF(n_components=n_factors, random_state=seed, init="nndsvd")

In [ ]:
Z_rna = nmf.fit_transform(visium.X)
loadings_rna = nmf.components_

print(Z_rna.shape, loadings_rna.shape)

for factor in range(n_factors):
    visium.obs[f"NMF_Z_{factor + 1}"] = Z_rna[:, factor]

We have run NMF on the Visium view. As output we get the learned embedding, e.g. factor matrix Z, which is n_obs x n_factors and the corresponding loadings with shape n_factors x n_features (rna). We can save the embeddings to our anndata object and then in the next step, use the ```sc.pl.spatial()``` function to overlay the H&E embedding with each learned factor.

In [ ]:
sc.pl.spatial(visium, color=[f"NMF_Z_{factor + 1}" for factor in range(n_factors)], color_map="Blues", alpha_img=0.3)

Visually, we can identify that factor 3, corresponds to the dopaminergic regions, however, we can also quantify this observation by computing the correlation between a known feature from the MSI-view, e.g. mz-390.16864, by computing the cross-correlation to each of the factors. Depending on your dataset and research question, you can already pindown a variety of questions. For instance the authors of MAGPIE (Williams et al., 2026), generated VISIUM and DESI-MSI data, investigating the drug distribution (AZX compound) within the tissue and the resulting changes, where they exactly did this. They have feature of interest, in one view and look for co-localization in the other view, to link the mechanistic effect of a certain drug on the transcriptome.

In [ ]:
# compute cross correlation

dopamine = msi[:, "mz-390.16864"].X.toarray()

correlations = []
for factor in range(n_factors):
    correlations.append(pearsonr(dopamine.flatten(), Z_rna[:, factor])[0])

correlations = np.array(correlations)

factors = np.arange(len(correlations))
factor_labels = [str(i + 1) for i in factors]

sorted_indices = np.argsort(correlations)[::-1]
sorted_correlations = correlations[sorted_indices]
sorted_labels = [factor_labels[i] for i in sorted_indices]

fig, ax = plt.subplots(figsize=(4, 5))

bars = ax.barh(range(len(sorted_correlations)), sorted_correlations)

ax.set_yticks(range(len(sorted_correlations)))
ax.set_yticklabels(sorted_labels)
ax.set_xlabel("Spatial cross-correlation\nto Dopamine (mz-390.16864)")
ax.set_ylabel("Visium NMF factors")
ax.invert_yaxis()
ax.grid(axis="x", alpha=0.3)
plt.show()

**Task 2:** Retrieve the top 10 features for factor 3. Check their biological functions.

### NMF on MSI view


Previously, we have computed NMF for the RNA view and could link a Dopamine (m/z) to a factor. Now we will compute NMF on the MSI view and check whether we learn similar pattern, by computing the cross-correlation between our Z_rna and Z_msi.

In [ ]:
Z_msi = nmf.fit_transform(msi.X)
loadings_msi = nmf.components_

for factor in range(n_factors):
    msi.obs[f"NMF_Z_msi_{factor + 1}"] = Z_msi[:, factor]

**Task 2**: Compute NMF on the MSI view and save the learned embedding in the MSI anndata object. Plot the embeddings on top of the H\&E, as in the previous section.

**Task 3**: Do we learn similar NMF patterns from both views? E.g. can we link them directly by matching the factors with the highest correlation? Btw. this is an analysis done by Godfrey et al., the computed for both modalities NMF respectively and then paired linked up the factors post-hoc, to identify features from both modalities that co-vary along the same spatial axis (pattern).

Based on the previous example, you could identify correlated patterns from both learned embeddings. Now you can select them and plot their top loadings. **Optional**: plot their expression, do they actually share an expression patterns as defined the learned pattern?

In [ ]:
df_rna = pd.DataFrame(
    {
        "feature": visium.var_names,
        "loading": loadings_rna[3 - 1, :],  # Factor 3
    }
)

df_msi = pd.DataFrame(
    {
        "feature": msi.var_names,
        "loading": loadings_msi[3 - 1, :],  # Factor 3
    }
)


df_rna_sorted = df_rna.reindex(df_rna["loading"].abs().argsort()[::-1])
df_msi_sorted = df_msi.reindex(df_msi["loading"].abs().argsort()[::-1])

top_rna = df_rna_sorted.head(10)
top_msi = df_msi_sorted.head(10)


fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 5))

y_pos_rna = np.arange(len(top_rna))
bars1 = (ax1.barh(y_pos_rna, top_rna["loading"]),)  # color='')
ax1.set_yticks(y_pos_rna)
ax1.set_yticklabels(top_rna["feature"])
ax1.set_xlabel("Loadings")
ax1.set_title("Top 10 RNA-seq features (factor 3)")
ax1.invert_yaxis()
ax1.grid(axis="x", alpha=0.3)

y_pos_msi = np.arange(len(top_msi))
bars2 = ax2.barh(y_pos_msi, top_msi["loading"])  # , color='orange')
ax2.set_yticks(y_pos_msi)
ax2.set_yticklabels(top_msi["feature"])
ax2.set_xlabel("Loadings")
ax2.set_title("Top 10 MSI features (factor 3)")
ax2.invert_yaxis()
ax2.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

**Task 3**: Which other learned factor in both modalities is highly correlated? Plot the factor for both and the corresponding loadings in both views. What do you think about this approach? 

### Optional: Pre-joint (by concatenation) learning

We are almost there, one step away from using MOFA/MEFISTO. As discussed in the lecture concatenation and then applying a method on the concatenated view is also a way to integrate multi-modal data.

**Task 4: what are the drawbacks again of the naive concatenation approach?**

In [ ]:
X = np.concatenate([visium.X.toarray(), msi.X.toarray()], axis=1)
Z_cat = nmf.fit_transform(X)
W_cat = nmf.components_

for factor in range(n_factors):
    visium.obs[f"NMF_Z_cat{factor + 1}"] = Z_cat[:, factor]
sc.pl.spatial(visium, color=[f"NMF_Z_cat{factor + 1}" for factor in range(n_factors)], color_map="Blues", alpha_img=0.4)

**Task 5** 1) Plot the loadings for Factor 4 and 7. 2) Optional: plot the expression of the top features. 3) Compare it with loadings learned in the separate approach (e.g. retrieve all and compute the pearson correlation), can we learn the same loadings or is joint learning helping us discovering something different?

**Task 6**: Look up genes and/ or m/z ratios (use the supps provided in the data folder) of your choice. You can play around with plotting different factors and look at their top loadings for both views.

## Learning spatial patterns jointly using MOFA/MEFISTO

In this exercise, we will use the 1:1 matched data, i.e., ```visium``` and ```msi_aggregated``` datasets, and train a MOFA model. MOFA applies a univariate prior on the factors (observations are i.i.d.). Later, you can compare this with MEFISTO, which uses a Gaussian process prior, where observations are not i.i.d. but spatially autocorrelated.

Compared to our previous NMF analysis, MOFA produces real-valued factors and loadings. Furthermore, it learns latent factors from a multi-modality (multi-view) perspective in a principled manner, capturing shared and modality-specific variation across datasets, which we can identify by investigating the variance explained values

For MOFA, the data must be normalized by library size, and the features should be scaled to have a mean of 0 and a standard deviation of 1.

In [ ]:
# preprocess: library size normalization and then per feature scaling (mean=0, sd=1)
visium_mofa = visium.copy()
visium_mofa.X = visium_mofa.layers["counts"].copy()
sc.pp.normalize_total(visium_mofa, target_sum=1e4)
sc.pp.log1p(visium_mofa)
sc.pp.scale(visium_mofa)


msi_mofa = msi.copy()
msi_mofa.X = msi_mofa.layers["raw"].copy()
sc.pp.normalize_total(msi_mofa, target_sum=1e4)  # total ion count norm
sc.pp.log1p(msi_mofa)
sc.pp.scale(msi_mofa)

In [ ]:
mofaplus = entry_point()

# data structure [[modality1], [modality2]] within modality1 [group1, group2, ...]
# note: we have only one group (group1) here, a group in our terms is just a tissue section or subject

mofaplus.set_data_matrix(
    data=[[visium_mofa.X], [msi_mofa.X]],
    views_names=["visium", "msi"],
    groups_names=["group1"],
    features_names=[visium_mofa.var_names.tolist(), msi_mofa.var_names.tolist()],
)

mofaplus.set_model_options(
    factors=n_factors,
    spikeslab_factors=False,
    spikeslab_weights=True,
    ard_factors=True,
    ard_weights=True,
)

mofaplus.set_train_options(
    iter=1000,
    convergence_mode="fast",
    gpu_mode=False,
    gpu_device=None,
    seed=seed,
)

mofaplus.build()
mofaplus.run()

# save our model
mofaplus.save("data/mofa.hdf5")

In [ ]:
# if youre unnable to run the above code, you can load the pre-trained model here:
# mofa = mfx.mofa_model("data/mofa.hdf5")
# MOFA_Z = mofa.get_factors()
# MOFA_loadings = mofa.get_weights(views=0) # 0 for visium, 1 for msi
# print(MOFA_Z.shape, MOFA_loadings.shape)

In [ ]:
MOFA_loadings = mofaplus.model.nodes["W"].getExpectation()
MOFA_Z = mofaplus.model.nodes["Z"].getExpectation()

# save in AnnData
for factor in range(n_factors):
    visium.obs[f"MOFA_Z_{factor + 1}"] = MOFA_Z[:, factor]
    msi.obs[f"MOFA_Z_{factor + 1}"] = MOFA_Z[:, factor]

**Task 7**: Plot the learned MOFA factors with ```sc.pl.spatial()``` function as previously. Which factors seems to be associated with dopaminergic neurons?

**Task 8**: Factor 4 seems to be representing the dopamin associated region. Plot the loadings for each of the modalities and look at their top loadings (Hint: use the code from previous section and adapt it). What do you observe? Have a closer look at features of your interest.

### No more custom plotting functions to write


Writing custom functions to plot results of your factor analysis is time consuming and inefficient. MOFAX is a convenient plotting package for MOFA/MEFISTO outputs, that you can just use.

**Task 9**:  Use [mofax](https://github.com/bioFAM/mofax). Have a look of different plots and their implications. Get inspiration through this [notebook](https://github.com/bioFAM/mofax/blob/master/notebooks/getting_started_pbmc10k.ipynb) and just plot.

In [ ]:
mofa = mfx.mofa_model("data/mofa.hdf5")

# always remember to close the model, otherwise you can't rewrite the file (e.g. to save a new model with the same name)
# mofa.close()

In [ ]:
mofa.close()

**Task 10**: Previously we had only positive loadings but in MOFA you can also have negative loadings. How can you interpret them?

## Second part, including spatial information

Use [MEFISTO](https://www.nature.com/articles/s41592-021-01343-9) to learn spatial patterns. MEFISTO uses a multivariate prior (Gaussian process) to model observation-observation dependencies. Compare with the MOFA and NMF results. If you want to use MEFISTO, it is a seaminglessy transition, you just need to define your spatiali coordinates and setup ```set_smooth_options``` and then it is just plug in play. We will use sparse Gaussian processes, i.e. use a set of inducing points to approximate the covariance matrix. MEFISTO takes around 3 min on my comoputer. If it takes too long, just load the MEFISTO model using mofax.

In [ ]:
mofaplus = entry_point()

spatial_coords = visium_mofa.obsm["spatial"].copy()


mofaplus.set_data_matrix(
    data=[[visium_mofa.X], [msi_mofa.X]],
    views_names=["RNA", "MSI"],
    groups_names=["group1"],
    features_names=[
        visium_mofa.var_names.tolist(),
        msi_mofa.var_names.tolist(),
    ],
)


mofaplus.set_covariates(sample_cov=[spatial_coords], covariates_names=["imagerow", "imagecol"])


n_inducing = 1000
n_obs = visium_mofa.n_obs

mofaplus.set_model_options(
    factors=n_factors,
    spikeslab_factors=False,  # not supported for MEFISTO
    spikeslab_weights=True,
    ard_factors=False,  # not supported for MEFISTO
    ard_weights=True,
)

mofaplus.set_train_options(
    iter=1000,
    convergence_mode="fast",
    gpu_mode=False,
    gpu_device=None,
    seed=seed,
)

mofaplus.set_smooth_options(
    sparseGP=True,
    warping=False,
    start_opt=10,
    opt_freq=10,
    frac_inducing=n_inducing / n_obs,
)

mofaplus.build()
mofaplus.run()
mofaplus.save("data/mefisto.hdf5")

In [ ]:
# if youre unnable to run the above code, you can load the pre-trained model here:
# mefisto = mfx.mofa_model("mefisto.hdf5")
# MEFISTO_Z = mefisto.get_factors()
# MEFISTO_loadings = mefisto.get_weights(views=0) #f 0 for visium, 1 for msi
# print(MEFISTO_Z.shape, MEFISTO_loadings.shape)

In [ ]:
MEFISTO_loadings = mofaplus.model.nodes["W"].getExpectation()
MEFISTO_Z = mofaplus.model.nodes["Z"].getExpectation()

# save in AnnData
for factor in range(n_factors):
    visium.obs[f"MEFISTO_Z_{factor + 1}"] = MEFISTO_Z[:, factor]
    msi.obs[f"MEFISTO_Z_{factor + 1}"] = MEFISTO_Z[:, factor]

In [ ]:
sc.pl.spatial(
    visium, color=[f"MEFISTO_Z_{factor + 1}" for factor in range(n_factors)], color_map="RdBu", vcenter=0, alpha_img=0.1
)

**Task 11**: How do the factors compare visually to MOFA factors? Plot one factor, e.g. pattern covering the dopamineergic neurons from MOFA/and MEFISTO one by one. 

**Task 12**: Plot the figures using MOFAX that you're interested in, e.g. variance explained and factor loadings? 

In [ ]:
mefisto = mfx.mofa_model("data/mefisto.hdf5")

# remember to close it after your finished, otherwise you won't be able to overwrite it with the next run
# mefisto.close()

In [ ]:
mefisto.close()

**Task 13**: Do you see a difference between non-negative vs. real-valued analysis, and separate vs. joint analysis of spatial multi-omics data?

**Task 14**: What kinds of downstream analyses, other than examining the loadings, could you imagine?

# Summary

We covered a lot of ground in this session. The analyses demonstrated here are meant as inspiration — a starting point for deeper investigation or a thread you can follow and explore further. We looked at the separate analyses grounded in the biological knowledge from the experiment, and at joint analyses using both a naive approach and a more principled one with MOFA/MEFISTO. MEFISTO uses a Gaussian process prior, so the spatial coordinates that are kept untouched in the previous analysis - are used in MEFISTO to spatially inform the learnt factors.

One more note: we worked on a single subject/slide here, but MOFA/MEFISTO makes it easy to add more groups. You can use this to identify structure across conditions (e.g. healthy vs. disease) or simply to increase power by adding biological replicates.

For further questions — or consulting on your own datasets — we're always happy to discuss.

# Optional task

For the ones, who are super fast or are interested after the summer school to explore one section (```BC515_2```) from [Godfrey et al.](https://onlinelibrary.wiley.com/doi/full/10.1002/anie.202502028), who applied DESI-MSI followed by Visium on the same slide for 3 breast cancer samples and 3 lung cancer samples. In the folder ```godfrey``` you will find the anndata object for BC515 (they have two replicates of this because on tissue folded during prep). 

In the paper they reported following feature pairs to be highly correlated:

* glycerophosphorylethanolamine (GPEA) and X-box binding protein 1 (XBP1) (BC1)
* vitamin B5 and adhesion G proteincoupled receptor B2 (ADGRB2) (BC1.2)
* GPEA and the Forkhead box protein A1 (FOXA1) (BC2)
* PI 18:0_18:1 and the Chemokine (C-X-C motif) ligand 14 (CXCL14) (BC3)

You could use this as anchor points for you analysis. You can find the m/z - metabolite annotations in the ```supps.xlsx``` file.

In [ ]:
import anndata as ad

BC = ad.read_h5ad("data/godfrey2025/BC_515_Section_2.h5ad")

**Task 1**: Where is MSI data? 

**Task 2**: Plot the pathologist annotation saved in ```obs["annotation"]``` (you could use this as anchor point for your analysis, which learned patterns are correlated with which pathological annotation)

**Task 3**: Conduct an analysis of your choice on this subject (If you apply MOFA, check for annotation based plotting, e.g. by cell type or in our case pathological annotation)